# Kaggle Notebook Setup - Qwen3.5-4B + Al-Nisyan Memory System
Run this notebook on Kaggle with GPU (T4) enabled.

In [ ]:
# Cell 1: Install dependencies
!pip install -q unsloth transformers accelerate bitsandbytes torch

In [ ]:
# Cell 2: Prepare repo path (clone or uploaded dataset)
import os
import sys
import subprocess
import shutil
from pathlib import Path

# IMPORTANT: put your real public GitHub repo URL here if repo is not uploaded as Kaggle Input
REPO_URL = os.getenv("AL_NISYAN_REPO_URL", "https://github.com/faresrafat3/al-nisyan.git").strip()
working_repo = Path("/kaggle/working/al-nisyan")
input_root = Path("/kaggle/input")
cwd_repo = Path.cwd()

def is_repo_root(path: Path) -> bool:
    return (path / "src").exists() and (path / "experiments").exists()

candidates = [working_repo, cwd_repo]
if input_root.exists():
    candidates.extend([p for p in input_root.iterdir() if p.is_dir()])
project_root = None
for candidate in candidates:
    if is_repo_root(candidate):
        project_root = candidate
        break
    for nested in candidate.glob("**/*") if candidate.exists() else []:
        if nested.is_dir() and is_repo_root(nested):
            project_root = nested
            break
    if project_root is not None:
        break

if project_root is not None and project_root != working_repo:
    if working_repo.exists():
        shutil.rmtree(working_repo)
    print(f"Copying project to writable path: {working_repo}")
    shutil.copytree(project_root, working_repo)
    project_root = working_repo
elif project_root is not None:
    pass
elif REPO_URL and "<YOUR_USERNAME>" not in REPO_URL and "your-repo" not in REPO_URL:
    print("Cloning repository into /kaggle/working/al-nisyan ...")
    subprocess.run(["git", "clone", REPO_URL, str(working_repo)], check=True)
    project_root = working_repo
elif working_repo.exists():
    project_root = working_repo
else:
    raise RuntimeError(
        "Repository not found. Set REPO_URL to your real public GitHub repo, or upload the project as Kaggle Input."
    )

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Has src: {(project_root / 'src').exists()}")
print(f"CWD writable: {os.access(project_root, os.W_OK)}")

In [ ]:
# Cell 3: Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Cell 4: Load model
from src.models.memory_augmented_model_v2 import CultivatedMemoryModel
Cell 4 diagnostic: hidden-dim resolver loaded from local source

model = CultivatedMemoryModel(
    base_model_key="Qwen/Qwen3.5-4B",
    memory_slots=1024,
    memory_dim=512,
    controller_mode="clean",
)

print(f"VRAM used: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Model: {model.loader.get_model_info()}")

In [ ]:
# Cell 5: Run official experiment
from experiments.forgetting_experiment import run_forgetting_experiment

results = run_forgetting_experiment(
    model_key="Qwen/Qwen3.5-4B",
    memory_slots=1024,
    save_results=True,
)

print(f"\nFinal Stats: {model.get_garden_stats()}")